# 🏠 Arquitetura Medallion - Visão Geral

Este notebook executa o pipeline real do projeto nas 3 camadas:

1. **Bronze**: ingestão dos CSVs brutos (`data/`) para tabelas Delta
2. **Silver**: limpeza, tipagem e normalização
3. **Gold**: agregações de negócio

Stack: Spark + Delta Lake (+ MinIO opcional com `USE_MINIO=1`).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.session import get_spark
from src.ingestion.Bronze import run as run_bronze
from src.processing.Silver import run as run_silver
from src.serving.Gold import run as run_gold

spark = get_spark("MedallionOverview")
print("Spark Session criada com sucesso!")

## 1. Camada BRONZE — Ingestão de Dados Brutos

In [ ]:
run_bronze(spark)
spark.table("bronze.orders").show(5, truncate=False)

## 2. Camada SILVER — Limpeza e Tipagem

In [ ]:
run_silver(spark, preview=True)

## 3. Camada GOLD — Agregações de Negócio

In [ ]:
run_gold(spark, show=True)

## 4. Consultas nas tabelas Gold

In [ ]:
print("=== Vendas por Categoria ===")
spark.table("gold.vendas_por_categoria").show(truncate=False)

print("=== Pedidos por Status ===")
spark.table("gold.pedidos_por_status").show(truncate=False)

In [ ]:
print("=== Top 10 Clientes por Gasto ===")
spark.table("gold.resumo_clientes").limit(10).show(truncate=False)

In [ ]:
print("=== SQL: receita total por status ===")
spark.sql('''
    SELECT status,
           SUM(receita_total) AS receita,
           SUM(total_pedidos) AS pedidos
    FROM gold.pedidos_por_status
    GROUP BY status
    ORDER BY receita DESC
''').show(truncate=False)

## 5. Histórico de Transações Delta

In [ ]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "bronze.orders")
history = delta_table.history()

print("=== Histórico (bronze.orders) ===")
history.select(
    "version",
    "timestamp",
    "operation",
    "numOutputRows",
).show(truncate=False)

In [ ]:
spark.stop()
print("\nSessão Spark encerrada.")